## Abstract

Despite the header being `## Abstract`, this section will render as a highlighted section titled *Summary*. Ensure this section is a **maximum** of 280 characters.

----

:::{.callout-note title="AI usage disclosure" collapse="true"}
This is a placeholder for the AI usage disclosure. Once all authors sign the AI code form on Airtable, SlackBot will message you an AI disclosure that you should place here.
:::

## Purpose

Once edited by you, this file will become your publication. Alternatively, if you already have a notebook written that you're trying to transform into a pub, replace this file with your own, but be sure to add the YAML front matter (the first cell) to your notebook.

Your pub should begin with a section titled **Purpose** where you, as briefly as possible, explain why you did the work described in the pub, the key takeaway, your primary audience, and how you think it could be useful to them/why you're sharing it.

## Learn more

Refer to [Demo](/examples/demo.html) to learn about possible syntax.

## Demo: single-cell segmentation at scale

The cells below run the full pipeline end to end on a handful of example images: brightfield ND2s are read from S3, segmented with Cellpose-SAM (via [`arcadia-microscopy-tools`](https://github.com/Arcadia-Science/arcadia-microscopy-tools)), and the derived label masks, QC overlays, and per-cell measurement tables are written back to S3. All the heavy GPU compute happens on [Modal](https://modal.com/); the notebook itself only triggers the run and displays small previews returned by the workers.

Running this section requires two one-time setup steps: authenticate the Modal client locally with `uv run modal setup`, and create a Modal secret named `aws-s3-access-secret` holding AWS credentials with read/write access to the `cloud-scaled-cell-phenotyping` bucket. The segmentation runs on paid GPU containers, so it is scoped to a few files here via `limit`.

In [ ]:
from analysis.runner import run_segmentation

# The example dataset: single-channel brightfield ND2s under `datasets/<DATASET_ID>/raw/`.
DATASET_ID = "20260731_140641_821"

# Segment the first few images. Full artifacts (masks, overlays, measurement CSVs) are
# written back to S3 under `datasets/<DATASET_ID>/derived/{masks,overlays,measurements}/`.
run = run_segmentation(DATASET_ID, limit=3)

print(f"Wrote artifacts to s3://cloud-scaled-cell-phenotyping/{run.output_prefix}/")
print(f"Segmented {len(run.results)} image(s), {len(run.failures)} failure(s).")

### Segmentation overlays

Each worker returns a small outline-overlay preview (cell boundaries drawn on the brightfield image), which we display here as a quick visual QC of the segmentation.

In [ ]:
from pathlib import Path

from IPython.display import Image as IPyImage
from IPython.display import display

for summary in run.results:
    overlay_path = summary.get("local_overlay")
    if overlay_path:
        print(f"{Path(summary['input']).name}: {summary['n_cells']} cells")
        display(IPyImage(filename=overlay_path, width=500))

### Per-cell morphology

The workers also return per-cell measurements (regionprops, converted to microns where the pixel size is known). We pool them across the demo images and plot a couple of shape features to show the kind of phenotyping this pipeline enables.

In [ ]:
import matplotlib.pyplot as plt

measurements = run.measurements
measurements.head()

In [ ]:
# Cell area (in microns^2 when the pixel size is available, else pixels) vs. eccentricity.
area_col = "area_um2" if "area_um2" in measurements.columns else "area"

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(measurements[area_col], measurements["eccentricity"], s=8, alpha=0.5)
ax.set_xlabel("cell area (um^2)" if area_col == "area_um2" else "cell area (px)")
ax.set_ylabel("eccentricity")
ax.set_title(f"Per-cell morphology (n = {len(measurements)} cells)")
plt.show()